In [1]:
# Import các thư viện
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image
import os
import torch


In [2]:
# Cấu hình đường dẫn dataset
dataset_path = "./data_detect/dataset.yaml"

# Kiểm tra dataset
print("Dataset configuration:")
with open(dataset_path, 'r', encoding='utf-8') as f:
    print(f.read())

# Kiểm tra số lượng ảnh training và validation
train_images = len([f for f in os.listdir("data_detect/images/train") if f.endswith('.png', )])
val_images = len([f for f in os.listdir("data_detect/images/val") if f.endswith('.png')])
train_labels = len([f for f in os.listdir("data_detect/labels/train") if f.endswith('.txt')])

print(f"\nDataset statistics:")
print(f"Training images: {train_images}")
print(f"Validation images: {val_images}")
print(f"Training labels: {train_labels}")


Dataset configuration:
train: /Users/hoangtranminh/Documents/python/model_dectect/data_detect/images/train/
val: /Users/hoangtranminh/Documents/python/model_dectect/data_detect/images/val/
nc: 2
names: ['BSD', 'BSV']
#/Users/hoangtranminh/Documents/python/model_dectect/data_detect/images/train

Dataset statistics:
Training images: 3433
Validation images: 1145
Training labels: 2830


In [ ]:
# Fine-tune YOLOv8 model
def train_yolov8():
    # Load pre-trained YOLOv8 model
    model = YOLO('yolov8n.pt')  # Bạn có thể thay đổi thành yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt
    
    # Training parameters
    results = model.train(
        data=dataset_path,
        epochs=100,
        imgsz=640,
        batch=16,
        lr0=0.01,
        patience=20,
        project='runs/detect',
        name='license_plate_detection_min'
    )
    
    return model, results

# Bắt đầu training
print("Bắt đầu training YOLOv8 model...")
model, training_results = train_yolov8()
print("Training hoàn thành!")


In [6]:
# Load trained model và inference
def load_trained_model():
    # Load model đã được train (thay đổi đường dẫn nếu cần)
    model_path = "./runs/detect/license_plate_detection_min/weights/best.pt"
    
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"Đã load model từ: {model_path}")
    else:
        # Nếu chưa có model trained, sử dụng model pre-trained
        model = YOLO('yolov8n.pt')
        print("Sử dụng model pre-trained (chưa fine-tune)")
    
    return model

# Load model
model = load_trained_model()


Đã load model từ: ./runs/detect/license_plate_detection_min/weights/best.pt


In [8]:
# Vẽ biểu đồ Training & Validation Loss
import pandas as pd
from pathlib import Path

def plot_training_validation_loss(results_dir="./runs/detect/license_plate_detection_min"):
    """
    Vẽ biểu đồ Training vs Validation Loss đơn giản
    """
    results_path = Path(results_dir)
    csv_file = results_path / "results.csv"
    
    if not csv_file.exists():
        print(f"❌ Không tìm thấy file results.csv tại: {csv_file}")
        print("Hãy chạy training trước!")
        return
    
    # Đọc dữ liệu
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.strip()
    
    # Vẽ biểu đồ
    plt.figure(figsize=(10, 6))
    
    if 'train/box_loss' in df.columns:
        plt.plot(df['epoch'], df['train/box_loss'], 'b-', linewidth=2, label='Training Loss')
    
    if 'val/box_loss' in df.columns:
        plt.plot(df['epoch'], df['val/box_loss'], 'orange', linewidth=2, label='Validation Loss')
    
    plt.title('Biểu đồ Training & Validation Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Vẽ biểu đồ
plot_training_validation_loss()

<Figure size 1000x600 with 1 Axes>

In [4]:
import time
from ultralytics import YOLO

model = YOLO("./runs/detect/license_plate_detection_min/weights/best.pt")

# Đánh giá hiệu suất mô hình
def evaluate_model(model, val_data_path="./data_detect/images/val"):
    """
    Đánh giá hiệu suất mô hình trên validation set + đo tốc độ xử lý
    """
    print("Bắt đầu đánh giá mô hình...")
    
    # Validate model trên validation set
    results = model.val(data=dataset_path, split='val')

    # In kết quả đánh giá
    print("\n=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ===")
    print(f"mAP50: {results.box.map50:.4f}")
    print(f"mAP50-95: {results.box.map:.4f}")
    print(f"Precision: {results.box.mp:.4f}")
    print(f"Recall: {results.box.mr:.4f}")
    print(f"F1-Score: {2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr):.4f}")

    # --- Đo tốc độ xử lý ---
    test_img = val_data_path + "./data_detect/images/val/carlong_0050.png"  # chọn 1 ảnh trong tập val
    im = "./data_detect/images/val/carlong_0050.png" 
    img = im if im else None

    if img:
        start = time.time()
        preds = model.predict(img, imgsz=640, verbose=False)
        end = time.time()
        elapsed = end - start
        fps = 1 / elapsed
        print(f"\n=== TỐC ĐỘ XỬ LÝ ===")
        print(f"Thời gian cho 1 ảnh: {elapsed:.4f} giây")
        print(f"FPS (ảnh/giây): {fps:.2f}")
    
    return results

# Chạy đánh giá
evaluation_results = evaluate_model(model)


Bắt đầu đánh giá mô hình...
Ultralytics 8.3.203 🚀 Python-3.11.1 torch-2.8.0 CPU (Apple M4)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 628.0±294.9 MB/s, size: 184.5 KB)
val: Scanning /Users/hoangtranminh/Documents/python/model_dectect/data_detect/labels/val.cache... 1145 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1145/1145 4.1Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 0.7it/s 1:451.5sss
                   all       1145       1313       0.99      0.988      0.994      0.917
                   BSD        409        410      0.997      0.995      0.995      0.913
                   BSV        753        903      0.983       0.98      0.993       0.92
Speed: 0.3ms preprocess, 86.5ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /Users/hoangtranminh/Documents/python/model_dectect/runs/detect/

In [23]:
model = YOLO("./runs/detect/license_plate_detection_min/weights/best.pt")
# Hàm inference và hiển thị kết quả
def detect_license_plate(image_path, conf_threshold=0.5):
    """
    Detect biển số xe trong ảnh và hiển thị kết quả
    """
    # Đọc ảnh
    image = cv2.imread(image_path)
    if image is None:
        print(f"Không thể đọc ảnh: {image_path}")
        return None, None
    
    # Inference
    results = model(image, conf=conf_threshold)
    
    # Vẽ bounding box và label
    annotated_image = results[0].plot()
    
    # Lấy thông tin detection
    detections = []
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                # Lấy tọa độ bounding box
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                confidence = box.conf[0].cpu().numpy()
                class_id = int(box.cls[0].cpu().numpy())
                class_name = model.names[class_id]
                
                detections.append({
                    'bbox': [int(x1), int(y1), int(x2), int(y2)],
                    'confidence': float(confidence),
                    'class': class_name,
                    'class_id': class_id
                })
    
    return annotated_image, detections

In [44]:
import os
import cv2
import matplotlib.pyplot as plt
yolo_LP_detect = YOLO("./runs/detect/license_plate_detection_min/weights/best.pt") 



In [46]:
image_path="./image_video/test/xemayBigPlate89_jpg.rf.1d5b6bfaba97b07941be486336512a81.jpg"
img = cv2.imread(image_path)  # image_path là đường dẫn file ảnh
if img is None:
    raise ValueError(f"Không đọc được ảnh từ {image_path}")

plates = yolo_LP_detect.predict(source=img, imgsz=640, conf=0.6)


0: 480x640 1 BSV, 36.6ms
Speed: 1.9ms preprocess, 36.6ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)


In [47]:
import easyocr
# ... (các import khác như os, cv2, pandas, helper, utils_rotate) ...

# KHỞI TẠO EASYOCR READER CHỈ MỘT LẦN
READER = easyocr.Reader(['vi', 'en']) # Hỗ trợ Tiếng Việt và Tiếng Anh

In [49]:
import pandas as pd
from PIL import Image
import cv2
import torch
import math
import function.utils_rotate as utils_rotate
from IPython.display import display
import os
import function.helper as helper
import easyocr  # Thêm EasyOCR



# Giả sử:
# - plates đã có kết quả từ YOLO.
# - img là ảnh gốc.
# - helper.read_plate_from_crop là hàm OCR ban đầu của bạn.

if len(plates) > 0:
    # Lấy đối tượng Results đầu tiên
    result = plates[0]
    
    # Trích xuất dữ liệu và tạo DataFrame
    list_plates = pd.DataFrame(result.boxes.data.tolist(), columns=["xmin", "ymin", "xmax", "ymax", "confidence", "class_id"])
    list_plates = list_plates.values.tolist()
else:
    list_plates = []

list_read_plates = set()
count = 0

if len(list_plates) == 0:
    # Trường hợp không tìm thấy biển số bằng detection (YOLO)
    lp = helper.read_plate_from_crop(img, is_two_lines=False)
    if lp != "unknown":
        list_read_plates.add(lp)
else:
    for plate in list_plates:
        flag = 0
        x = int(plate[0])
        y = int(plate[1])
        w = int(plate[2] - plate[0])
        h = int(plate[3] - plate[1])
        cls_id = plate[5]
        
        # Đảm bảo kích thước crop hợp lệ
        if w <= 0 or h <= 0:
            continue
            
        crop_img = img[y:y + h, x:x + w]
        cv2.rectangle(img, (int(plate[0]), int(plate[1])), (int(plate[2]), int(plate[3])), color=(0, 0, 225), thickness=2)
        
        easyocr_result = READER.readtext(crop_img, detail=0)
        if easyocr_result:
            lp_easyocr = "".join(easyocr_result).replace(" ", "").upper()
            if lp_easyocr:
                list_read_plates.add(lp_easyocr)
                print(f"-> EasyOCR thành công! Biển số: {lp_easyocr}")
            else:
                print("-> EasyOCR không nhận dạng được văn bản.")
        else:
            print("-> EasyOCR không tìm thấy văn bản.")


-> EasyOCR thành công! Biển số: 65-81289.19
